<a href="https://colab.research.google.com/github/lgsepulveda/bioinformatics/blob/main/05_sequencing/exercises/01_3_genome_assembly_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧬 Práctica C: Ensamblaje de Genomas con Python + conda en Google Colab

**Dos bloques en un solo notebook:** Illumina (fastp + SPAdes) y Nanopore (NanoPlot + Filtlong + Flye). Usted elige el caso (01-20) y la tecnología con dos variables al inicio.

> ⚠️ **IMPORTANTE: antes de continuar**, lea la [guía de prácticas compartida](00_genome_assembly_common.md) — contiene el contexto biológico, los 20 casos de estudio (Illumina + Nanopore) y la tabla maestra de accesiones.

Este notebook cubre **ambos bloques tecnológicos**. Defina `CASO` (01-20) y `TECH` ("illumina" o "nanopore") en la celda de configuración (Paso 1) y el resto del notebook se adapta automáticamente: descarga los datos correctos, corre el pipeline de QC/limpieza correspondiente, ensambla con la herramienta adecuada (SPAdes para Illumina, Flye para Nanopore) y evalúa con QUAST.

Si quiere comparar ambas tecnologías para su caso asignado, ejecute el notebook completo una vez con `TECH = "illumina"`, guarde/descargue los resultados, luego cambie a `TECH = "nanopore"` y vuelva a ejecutar desde el Paso 1.


## ⚙️ Paso 0 — Configurar el entorno con conda

Ejecute esta celda **primero**. Instala Miniconda y todas las herramientas necesarias (ambos bloques).
⏱️ Puede tardar 8–12 minutos.

In [1]:
# Instalamos Conda para Colab

!pip install condacolab --quiet

import condacolab
condacolab.install() # Esto reinicia el kernel automáticamente


📢 Announcement 📢
condacolab==0.2 will be released soon! Try it with:

    !pip install -q https://github.com/conda-incubator/condacolab/archive/main.zip
    import condacolab
    condacolab.install()

0.2.x introduces a new installation method based on Pixi, with customizable Python versions.
This may be breaking for your workflow. If that's the case, please report it at
https://github.com/conda-incubator/condacolab and pin your `pip install` command to
condacolab==0.1 as a workaround.

⏬ Downloading https://github.com/conda-forge/miniforge/releases/download/26.3.2-3/Miniforge3-26.3.2-3-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:13
🔁 Restarting kernel...


Tras el reinicio del kernel, ejecuta esta celda para confirmar que el entorno quedó correctamente configurado y para dejar registradas las versiones de las herramientas base (útil para citar en la sección de métodos de informes/artículos).

In [ ]:
# Verificar instalación de condacolab y registrar versiones del entorno base
import os
import sys
import subprocess

os.environ['PATH'] = '/usr/local/bin:/opt/conda/bin:' + os.environ['PATH']

# Instalar biopython
!pip install biopython --quiet

def version_pip(paquete):
    res = subprocess.run(["pip", "show", paquete], capture_output=True, text=True)
    for linea in res.stdout.splitlines():
        if linea.startswith("Version:"):
            return linea.split(":", 1)[1].strip()
    return "⚠️ no encontrado"

conda_version = subprocess.run(["conda", "--version"], capture_output=True, text=True).stdout.strip()

# google-colab no se instala vía pip estándar en el entorno conda (es parte
# de la integración interna del runtime), por lo que 'pip show' no lo detecta
# de forma confiable. Verificamos disponibilidad importando el módulo.
try:
    import google.colab  # noqa: F401
    colab_estado = "disponible (ejecutando en Google Colab)"
except ImportError:
    colab_estado = "no detectado (¿no estás en Google Colab?)"

print("📦 Versiones del entorno base:")
print(f"  - Python: {sys.version.split()[0]}")
print(f"  - {conda_version}")
print(f"  - biopython: {version_pip('biopython')}")
print(f"  - Google Colab: {colab_estado}")

Una vez que Conda esté instalado, es posible que necesites reiniciar el entorno de ejecución de Colab (Runtime > Restart runtime) para que los cambios surtan efecto. Después de reiniciar, puedes ejecutar comandos `conda` en nuevas celdas de código.

### Instalar herramientas

Ya que hemos instalado **Conda** es necesario instalar las herramientas que vamos a usar para el ensamblaje.

In [ ]:
# Instalar herramientas bioinformáticas (bloque Illumina + bloque Nanopore + utilidades comunes)

# El log completo de conda es muy largo (incluye decenas de dependencias de
# soporte) y poco útil para reportes. Lo capturamos y, si todo sale bien,
# solo mostramos un resumen con las versiones de las herramientas pedidas
# (útil para citar versiones en informes/artículos). Si algo falla, se
# muestra el log completo para poder depurar.
import subprocess
import json

herramientas = [
    "sra-tools",
    "fastqc",
    "fastp",
    "spades",
    "nanoplot",
    "filtlong",
    "flye",
    "seqtk",
    "quast",
]

def mostrar_versiones_instaladas(paquetes):
    """Imprime una tabla compacta con nombre y versión de cada paquete instalado.
    Nota: 'conda list' solo acepta UN patrón regex como filtro, por eso se
    listan todos los paquetes del entorno y se filtra en Python."""
    res = subprocess.run(["conda", "list", "--json"], capture_output=True, text=True)
    try:
        instalados = {p["name"]: p["version"] for p in json.loads(res.stdout)}
    except (json.JSONDecodeError, KeyError):
        instalados = {}
    print("\n📦 Versiones instaladas:")
    for pkg in paquetes:
        print(f"  - {pkg}: {instalados.get(pkg, '⚠️ no encontrado')}")

print("Instalando herramientas bioinformáticas (esto puede tardar varios minutos)...")
cmd = ["conda", "install", "-y", "-c", "bioconda", "-c", "conda-forge"] + herramientas
resultado = subprocess.run(cmd, capture_output=True, text=True)

if resultado.returncode == 0:
    print("✅ Herramientas bioinformáticas instaladas")
    mostrar_versiones_instaladas(herramientas)
else:
    print("❌ Falló la instalación. Log completo de conda:\n")
    print(resultado.stdout)
    print(resultado.stderr)
    raise RuntimeError("La instalación de herramientas bioinformáticas falló. Revisa el log arriba.")

Ahora vamos a instalar otras herramientas propias de **Python** que nos ayudarán a procesar y ver los resultados más fácil para los humanos.

In [ ]:
!pip install matplotlib pandas --quiet

# Imports centralizados — todos en un solo lugar para evitar duplicación
import json
import os
import pathlib
import glob
import subprocess

import matplotlib.pyplot as plt
import pandas as pd

os.environ['PATH'] = '/opt/conda/bin:' + os.environ['PATH']

print("✅ Entorno listo (Illumina + Nanopore)")

## 📥 Paso 1 — Elegir caso y tecnología, y descargar los datos

Defina **dos variables**: el `CASO` (01-20, el mismo número asignado en el Módulo 4 de filogenética) y el `TECH` (`"illumina"` o `"nanopore"`).

Consulte la [tabla maestra de casos](00_genome_assembly_common.md#-tabla-maestra-de-casos) para ver la especie, el tamaño de genoma aproximado y si su caso está marcado como "pesado" (en ese caso, revise la sección de subsampleo antes de ensamblar).

In [ ]:
# ⚠️ **IMPORTANTE**: Defina el caso y la tecnología a trabajar aquí y solo aquí.
CASO = "01"          # <-- Número de caso, de "01" a "20" (mismo que en Módulo 4 de filogenética)
TECH = "illumina"    # <-- "illumina" o "nanopore"


In [ ]:
# Diccionario con los 20 casos: accesiones SRA (Illumina y Nanopore) + referencia NCBI para QUAST
CASOS_CONFIG = {
    "01": {
        "especie": "Klebsiella pneumoniae",
        "genoma_mb": 5.5,
        "illumina": {"acc": "ERR14828471", "ref_url": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/061/393/185/GCF_061393185.1_ASM6139318v1/GCF_061393185.1_ASM6139318v1_genomic.fna.gz"},
        "nanopore": {"acc": "SRR40944313", "ref_url": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/061/393/185/GCF_061393185.1_ASM6139318v1/GCF_061393185.1_ASM6139318v1_genomic.fna.gz"},
    },
    "02": {
        "especie": "Escherichia coli",
        "genoma_mb": 5.0,
        "illumina": {"acc": "SRR40503160", "ref_url": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/060/784/565/GCA_060784565.1_ASM6078456v1/GCA_060784565.1_ASM6078456v1_genomic.fna.gz"},
        "nanopore": {"acc": "SRR24837712", "ref_url": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/030/285/565/GCF_030285565.1_ASM3028556v1/GCF_030285565.1_ASM3028556v1_genomic.fna.gz"},
    },
    "03": {
        "especie": "Pseudomonas abieticivorans",
        "genoma_mb": 6.7,
        "illumina": {"acc": "SRR24684300", "ref_url": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/023/509/015/GCF_023509015.1_ASM2350901v1/GCF_023509015.1_ASM2350901v1_genomic.fna.gz"},
        "nanopore": {"acc": "SRR24684301", "ref_url": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/023/509/015/GCF_023509015.1_ASM2350901v1/GCF_023509015.1_ASM2350901v1_genomic.fna.gz"},
    },
    "04": {
        "especie": "Streptomyces venezuelae",
        "genoma_mb": 8.2,
        "illumina": {"acc": "SRR11960410", "ref_url": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/050/632/295/GCA_050632295.1_ASM5063229v1/GCA_050632295.1_ASM5063229v1_genomic.fna.gz"},
        "nanopore": {"acc": "SRR33220699", "ref_url": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/050/632/295/GCA_050632295.1_ASM5063229v1/GCA_050632295.1_ASM5063229v1_genomic.fna.gz"},
    },
    "05": {
        "especie": "Streptomyces coelicolor",
        "genoma_mb": 8.7,
        "illumina": {"acc": "SRR39165540", "ref_url": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/047/824/265/GCF_047824265.1_ASM4782426v1/GCF_047824265.1_ASM4782426v1_genomic.fna.gz"},
        "nanopore": {"acc": "SRR24425373", "ref_url": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/047/824/265/GCF_047824265.1_ASM4782426v1/GCF_047824265.1_ASM4782426v1_genomic.fna.gz"},
    },
    "06": {
        "especie": "Bacillus subtilis",
        "genoma_mb": 4.2,
        "illumina": {"acc": "ERR15846824", "ref_url": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/982/518/465/GCF_982518465.1_NRS2125/GCF_982518465.1_NRS2125_genomic.fna.gz"},
        "nanopore": {"acc": "SRR40784809", "ref_url": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/982/518/465/GCF_982518465.1_NRS2125/GCF_982518465.1_NRS2125_genomic.fna.gz"},
    },
    "07": {
        "especie": "Staphylococcus aureus (MRSA)",
        "genoma_mb": 2.8,
        "illumina": {"acc": "DRR187559", "ref_url": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/982/303/505/GCF_982303505.1_24S859_ONT_hybrid/GCF_982303505.1_24S859_ONT_hybrid_genomic.fna.gz"},
        "nanopore": {"acc": "SRR40597729", "ref_url": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/982/303/505/GCF_982303505.1_24S859_ONT_hybrid/GCF_982303505.1_24S859_ONT_hybrid_genomic.fna.gz"},
    },
    "08": {
        "especie": "Vibrio cholerae",
        "genoma_mb": 4.0,
        "illumina": {"acc": "SRR40968872", "ref_url": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/055/797/365/GCF_055797365.1_ASM5579736v1/GCF_055797365.1_ASM5579736v1_genomic.fna.gz"},
        "nanopore": {"acc": "ERR17674410", "ref_url": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/055/797/365/GCF_055797365.1_ASM5579736v1/GCF_055797365.1_ASM5579736v1_genomic.fna.gz"},
    },
    "09": {
        "especie": "Mycobacterium tuberculosis",
        "genoma_mb": 4.4,
        "illumina": {"acc": "ERR17981994", "ref_url": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/061/392/705/GCF_061392705.1_ASM6139270v1/GCF_061392705.1_ASM6139270v1_genomic.fna.gz"},
        "nanopore": {"acc": "SRR40948678", "ref_url": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/061/392/705/GCF_061392705.1_ASM6139270v1/GCF_061392705.1_ASM6139270v1_genomic.fna.gz"},
    },
    "10": {
        "especie": "Agrobacterium tumefaciens",
        "genoma_mb": 5.6,
        "illumina": {"acc": "DRR1119936", "ref_url": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/047/731/285/GCA_047731285.1_ASM4773128v1/GCA_047731285.1_ASM4773128v1_genomic.fna.gz"},
        "nanopore": {"acc": "SRR31799201", "ref_url": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/047/731/285/GCA_047731285.1_ASM4773128v1/GCA_047731285.1_ASM4773128v1_genomic.fna.gz"},
    },
    "11": {
        "especie": "Deinococcus radiodurans",
        "genoma_mb": 3.3,
        "illumina": {"acc": "SRR37705175", "ref_url": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/045/277/105/GCF_045277105.1_KDR13939K_1.0/GCF_045277105.1_KDR13939K_1.0_genomic.fna.gz"},
        "nanopore": {"acc": "ERR7055001", "ref_url": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/045/277/105/GCF_045277105.1_KDR13939K_1.0/GCF_045277105.1_KDR13939K_1.0_genomic.fna.gz"},
    },
    "12": {
        "especie": "Thermus thermophilus",
        "genoma_mb": 2.1,
        "illumina": {"acc": "SRR35352552", "ref_url": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/059/705/435/GCF_059705435.1_ASM5970543v1/GCF_059705435.1_ASM5970543v1_genomic.fna.gz"},
        "nanopore": {"acc": "ERR14102352", "ref_url": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/059/705/435/GCF_059705435.1_ASM5970543v1/GCF_059705435.1_ASM5970543v1_genomic.fna.gz"},
    },
    "13": {
        "especie": "Lactobacillus acidophilus",
        "genoma_mb": 2.0,
        "illumina": {"acc": "SRR38995089", "ref_url": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/988/235/355/GCF_988235355.1_IF3HA23006/GCF_988235355.1_IF3HA23006_genomic.fna.gz"},
        "nanopore": {"acc": "SRR34642242", "ref_url": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/988/235/355/GCF_988235355.1_IF3HA23006/GCF_988235355.1_IF3HA23006_genomic.fna.gz"},
    },
    "14": {
        "especie": "Helicobacter pylori",
        "genoma_mb": 1.6,
        "illumina": {"acc": "SRR40271341", "ref_url": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/059/997/335/GCA_059997335.1_ASM5999733v1/GCA_059997335.1_ASM5999733v1_genomic.fna.gz"},
        "nanopore": {"acc": "SRR40805640", "ref_url": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/059/997/335/GCA_059997335.1_ASM5999733v1/GCA_059997335.1_ASM5999733v1_genomic.fna.gz"},
    },
    "15": {
        "especie": "Salmonella enterica",
        "genoma_mb": 4.8,
        "illumina": {"acc": "SRR40979775", "ref_url": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/061/253/545/GCF_061253545.1_ASM6125354v1/GCF_061253545.1_ASM6125354v1_genomic.fna.gz"},
        "nanopore": {"acc": "SRR40942572", "ref_url": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/061/253/545/GCF_061253545.1_ASM6125354v1/GCF_061253545.1_ASM6125354v1_genomic.fna.gz"},
    },
    "16": {
        "especie": "Bacillus licheniformis",
        "genoma_mb": 4.3,
        "illumina": {"acc": "SRR40374058", "ref_url": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/055/397/075/GCA_055397075.1_ASM5539707v1/GCA_055397075.1_ASM5539707v1_genomic.fna.gz"},
        "nanopore": {"acc": "DRR1081654", "ref_url": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/055/397/075/GCA_055397075.1_ASM5539707v1/GCA_055397075.1_ASM5539707v1_genomic.fna.gz"},
    },
    "17": {
        "especie": "Paenibacillus polymyxa",
        "genoma_mb": 5.8,
        "illumina": {"acc": "SRR38851454", "ref_url": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/056/645/015/GCF_056645015.1_ASM5664501v1/GCF_056645015.1_ASM5664501v1_genomic.fna.gz"},
        "nanopore": {"acc": "SRR27678679", "ref_url": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/056/645/015/GCF_056645015.1_ASM5664501v1/GCF_056645015.1_ASM5664501v1_genomic.fna.gz"},
    },
    "18": {
        "especie": "Synechocystis sp.",
        "genoma_mb": 3.6,
        "illumina": {"acc": "ERR13636519", "ref_url": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/987/480/225/GCA_987480225.1_glgC6/GCA_987480225.1_glgC6_genomic.fna.gz"},
        "nanopore": {"acc": "ERR17229400", "ref_url": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/987/480/225/GCA_987480225.1_glgC6/GCA_987480225.1_glgC6_genomic.fna.gz"},
    },
    "19": {
        "especie": "Enterococcus faecalis",
        "genoma_mb": 3.0,
        "illumina": {"acc": "SRR40894698", "ref_url": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/061/255/735/GCA_061255735.1_ASM6125573v1/GCA_061255735.1_ASM6125573v1_genomic.fna.gz"},
        "nanopore": {"acc": "SRR40784437", "ref_url": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/061/255/735/GCA_061255735.1_ASM6125573v1/GCA_061255735.1_ASM6125573v1_genomic.fna.gz"},
    },
    "20": {
        "especie": "Shewanella oneidensis",
        "genoma_mb": 4.9,
        "illumina": {"acc": "SRR40344192", "ref_url": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/146/165/GCF_000146165.2_ASM14616v2/GCF_000146165.2_ASM14616v2_genomic.fna.gz"},
        "nanopore": {"acc": "SRR33906229", "ref_url": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/146/165/GCF_000146165.2_ASM14616v2/GCF_000146165.2_ASM14616v2_genomic.fna.gz"},
    },
}

if CASO not in CASOS_CONFIG:
    raise ValueError(f"⚠️ CASO '{CASO}' no reconocido. Use un número de \"01\" a \"20\".")
if TECH not in ("illumina", "nanopore"):
    raise ValueError(f"⚠️ TECH '{TECH}' no reconocido. Use \"illumina\" o \"nanopore\".")

cfg = CASOS_CONFIG[CASO]
tech_cfg = cfg[TECH]
acc = tech_cfg["acc"]
ref_url = tech_cfg["ref_url"]

print(f"── Caso {CASO} ({TECH}): {cfg['especie']} (~{cfg['genoma_mb']} Mb) ──")
print(f"Accesión SRA: {acc}")

workdir = f"genome_assembly/caso_{CASO}_{TECH}"
os.makedirs(f"{workdir}/data", exist_ok=True)

# --- Descargar lecturas con sra-tools (funciona igual para Illumina y Nanopore) ---
print("Descargando lecturas con prefetch...")
!prefetch "{acc}" -O {workdir}/data/ --max-size 50G

print("Extrayendo FASTQ con fasterq-dump...")
!fasterq-dump "{workdir}/data/{acc}/{acc}.sra" -O {workdir}/data/ --split-files -e 4

# Comprimir (fasterq-dump no comprime por defecto)
!gzip -f {workdir}/data/{acc}*.fastq

# Detectar si quedaron 1 o 2 archivos (Illumina paired-end = 2; Nanopore single-end = 1)
fastq_files = sorted(glob.glob(f"{workdir}/data/{acc}*.fastq.gz"))
print(f"Archivos FASTQ obtenidos: {fastq_files}")

if TECH == "illumina":
    r1_path = f"{workdir}/data/{acc}_1.fastq.gz"
    r2_path = f"{workdir}/data/{acc}_2.fastq.gz"
else:
    # Nanopore: un solo archivo de lecturas largas
    nano_path = fastq_files[0] if fastq_files else f"{workdir}/data/{acc}.fastq.gz"

# --- Descargar genoma de referencia (mismo para ambos bloques del caso, salvo Caso 02) ---
ref_gz = f"{workdir}/data/ref.fna.gz"
ref_path = f"{workdir}/data/ref.fna"
print("Descargando referencia NCBI...")
!wget -q "{ref_url}" -O "{ref_gz}"
!gunzip -f "{ref_gz}"

# Verificar tamaños de archivo
for fpath in (fastq_files + [ref_path]):
    size = os.path.getsize(fpath) if os.path.exists(fpath) else 0
    status = "✅" if size > 1000 else "❌ VACÍO o faltante"
    print(f"{status}  {fpath}  ({size:,} bytes)")

print(f"\n✅ Caso {CASO} ({TECH}) listo en {workdir}/")

## 👀 Paso 1.1 — Inspeccionar los archivos descargados

Vamos a verificar que los archivos FASTQ y el genoma de referencia FASTA se hayan descargado correctamente y tengan el formato esperado.

In [ ]:
if TECH == "illumina":
    print(f"=== Encabezado del archivo FASTQ R1 ({acc}) ===")
    !zcat "{r1_path}" | head -n 8
    print(f"\n=== Encabezado del archivo FASTQ R2 ({acc}) ===")
    !zcat "{r2_path}" | head -n 8
else:
    print(f"=== Encabezado del archivo FASTQ Nanopore ({acc}) ===")
    !zcat "{nano_path}" | head -n 8

print(f"\n=== Encabezado del archivo FASTA de referencia ===")
!head -n 5 "{ref_path}"

## 🔎 Paso 2 — Control de calidad y limpieza

Este paso usa una herramienta **distinta según la tecnología** — esto es intencional (ver nota en la guía compartida sobre por qué cada bloque usa herramientas diferentes):

- **Illumina:** `fastp` hace en un solo paso detección/recorte de adaptadores, filtrado por calidad Phred, y genera un reporte HTML + JSON.
- **Nanopore:** `NanoPlot` genera estadísticas y gráficas de calidad/longitud de las lecturas largas; luego `Filtlong` filtra las lecturas de peor calidad y descarta las muy cortas (no recorta bases individuales como fastp, porque Nanopore no tiene el concepto de "adaptador" por posición de la misma forma).

Parámetros clave:
- fastp `--qualified_quality_phred 30`: descarta bases con Phred < Q30; `--length_required 50`: descarta lecturas muy cortas tras el recorte.
- Filtlong `--min_length 1000`: descarta lecturas Nanopore menores a 1000 pb; `--keep_percent 90`: conserva el 90% de mejor calidad.

In [ ]:
os.makedirs(f"{workdir}/results/qc_results", exist_ok=True)

if TECH == "illumina":
    clean_r1 = f"{workdir}/results/clean_R1.fastq.gz"
    clean_r2 = f"{workdir}/results/clean_R2.fastq.gz"
    fastp_html = f"{workdir}/results/qc_results/fastp_report.html"
    fastp_json = f"{workdir}/results/qc_results/fastp_report.json"

    !fastp \
      -i "{r1_path}" -I "{r2_path}" \
      -o "{clean_r1}" -O "{clean_r2}" \
      --qualified_quality_phred 30 \
      --length_required 50 \
      --html "{fastp_html}" \
      --json "{fastp_json}" \
      --thread 4

    print(f"✅ fastp completo. Reporte: {fastp_html}")

else:
    nanoplot_out = f"{workdir}/results/qc_results/nanoplot"
    clean_nano = f"{workdir}/results/clean_nanopore.fastq.gz"

    # Paso 2a: QC con NanoPlot (reporte de longitud/calidad ANTES de filtrar)
    !NanoPlot --fastq "{nano_path}" -o "{nanoplot_out}" --plots dot --N50 -t 4

    # Paso 2b: Filtrado con Filtlong (descarta lecturas cortas/de baja calidad)
    # Si su caso está marcado "pesado" en la guía, puede agregar --target_bases <genoma_mb*1e6*80>
    !filtlong --min_length 1000 --keep_percent 90 "{nano_path}" | gzip > "{clean_nano}"

    print(f"✅ NanoPlot + Filtlong completos. Reporte NanoPlot: {nanoplot_out}/NanoPlot-report.html")

## 📊 Paso 3 — Explorar el reporte de QC con Python

- **Illumina:** leemos el JSON generado por `fastp` y mostramos un resumen comparativo antes/después del filtrado.
- **Nanopore:** leemos las estadísticas generadas por `NanoPlot` (`NanoStats.txt`).

In [ ]:
if TECH == "illumina":
    report = json.loads(pathlib.Path(fastp_json).read_text())

    summary = report["summary"]
    before  = summary["before_filtering"]
    after   = summary["after_filtering"]

    print("=== Resumen de calidad (fastp) ===")
    print(f"{'':30s} {'Antes':>12} {'Después':>12}")
    print("-" * 56)
    print(f"{'Lecturas totales':30s} {before['total_reads']:>12,} {after['total_reads']:>12,}")
    print(f"{'Bases totales':30s} {before['total_bases']:>12,} {after['total_bases']:>12,}")
    print(f"{'Q20 (%)':30s} {before['q20_rate']*100:>11.2f}% {after['q20_rate']*100:>11.2f}%")
    print(f"{'Q30 (%)':30s} {before['q30_rate']*100:>11.2f}% {after['q30_rate']*100:>11.2f}%")
    print(f"{'GC (%)':30s} {before['gc_content']*100:>11.2f}% {after['gc_content']*100:>11.2f}%")

else:
    nanostats_path = f"{workdir}/results/qc_results/nanoplot/NanoStats.txt"
    print("=== Resumen de calidad (NanoPlot, lecturas crudas) ===")
    !cat "{nanostats_path}"

### Comparación de Métricas de Calidad (solo bloque Illumina)

Graficamos el Phred score medio por posición en Read 1 y Read 2 a partir del reporte de `fastp`. Las zonas de color indican:
- 🔴 Rojo: Q < 20 (calidad baja, no confiable)
- 🟡 Amarillo: 20 ≤ Q < 30 (calidad media)
- 🟢 Verde: Q ≥ 30 (calidad alta, recomendada)

> Si `TECH == "nanopore"`, esta celda se salta — revise en su lugar las gráficas de `NanoPlot` (`NanoPlot-report.html`), que muestran longitud vs. calidad por lectura (no por posición).

In [ ]:
if TECH == "illumina":
    # se usa la curva 'mean' directamente del JSON de fastp
    def get_mean_quality_curve(read_section):
        """Extrae la curva de calidad media por posición del reporte fastp."""
        curves = read_section.get('quality_curves', {})
        mean_q = curves.get('mean', [])
        if not mean_q:
            return [], []
        return list(range(1, len(mean_q) + 1)), mean_q

    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
    pares = [
        ("read1_before_filtering", "read1_after_filtering", "Read 1"),
        ("read2_before_filtering", "read2_after_filtering", "Read 2"),
    ]
    for ax, (key_before, key_after, label) in zip(axes, pares):
        ax.axhspan(0, 20, color="red", alpha=0.08)
        ax.axhspan(20, 30, color="yellow", alpha=0.08)
        ax.axhspan(30, 45, color="green", alpha=0.08)

        x_before, y_before = get_mean_quality_curve(report.get(key_before, {}))
        if x_before:
            ax.plot(x_before, y_before, color="tab:blue", linestyle="--", linewidth=1.8, label="Antes del filtrado")

        x_after, y_after = get_mean_quality_curve(report.get(key_after, {}))
        if x_after:
            ax.plot(x_after, y_after, color="tab:green", linestyle="-", linewidth=1.8, label="Después del filtrado")

        ax.set_title(f"{label} — calidad media por posición (antes vs. después)")
        ax.set_xlabel("Posición en la lectura (pb)")
        ax.set_ylabel("Phred Score")
        ax.legend(loc="lower left")
    plt.tight_layout()
    plt.show()
else:
    print("⏭️  Celda omitida (solo aplica al bloque Illumina). Revise NanoPlot-report.html para el bloque Nanopore.")

## 🔧 Paso 4 — Ensamblaje de novo

- **Illumina:** `SPAdes` (St. Petersburg Assembler) reconstruye el genoma a partir de lecturas cortas paired-end usando grafos de De Bruijn.
- **Nanopore:** `Flye` reconstruye el genoma a partir de lecturas largas usando un enfoque de grafos de repetición, diseñado específicamente para las tasas de error más altas de Nanopore/PacBio.

Se usó una herramienta distinta en cada bloque **a propósito**: compare los resultados y note que ambas resuelven el mismo problema (ensamblar un genoma bacteriano) con estrategias algorítmicas diferentes.

In [ ]:
if TECH == "illumina":
    spades_out = f"{workdir}/results/spades_assembly"
    # Se usa --isolate para muestras bacterianas puras (recomendado)
    !spades.py \
      -1 "{clean_r1}" \
      -2 "{clean_r2}" \
      -o "{spades_out}" \
      --isolate \
      -t 4 \
      -m 12

    contigs_path = f"{spades_out}/contigs.fasta"

else:
    flye_out = f"{workdir}/results/flye_assembly"
    # --nano-raw asume lecturas Nanopore estándar (no corregidas con alta precisión / Q20+)
    # Si su run es de un kit reciente (Q20+/"super accuracy"), puede usar --nano-hq en su lugar
    !flye \
      --nano-raw "{clean_nano}" \
      --out-dir "{flye_out}" \
      --threads 4

    contigs_path = f"{flye_out}/assembly.fasta"

if os.path.exists(contigs_path):
    n_contigs = sum(1 for line in open(contigs_path) if line.startswith(">"))
    print(f"✅ Ensamblaje completo: {n_contigs} contigs en {contigs_path}")
else:
    print(f"❌ No se encontró el archivo de contigs esperado: {contigs_path}")

## 📏 Paso 5 — Evaluación del ensamblaje con QUAST

QUAST compara los contigs obtenidos (de SPAdes o Flye, según su `TECH`) contra el genoma de referencia para calcular métricas de calidad. Es el mismo paso para ambos bloques — esto permite comparar directamente los dos ensamblajes del mismo organismo.
Ver la definición de N50, L50 y Genome Fraction en las secciones **6** y **7.5** del [README del Módulo 5](../README.md).

In [ ]:
quast_out = f"{workdir}/results/quast_results"

!quast.py \
  "{contigs_path}" \
  -r "{ref_path}" \
  -o "{quast_out}" \
  --threads 4

print(f"✅ QUAST completo. Reporte en {quast_out}/report.html")

## 📊 Paso 6 — Leer y visualizar los resultados de QUAST con Python

Leemos el reporte TSV de QUAST y mostramos las métricas más importantes en una tabla y una gráfica de barras.

> 💡 Si ya corrió el otro bloque (`TECH`) para este mismo `CASO`, puede cargar ambos `report.tsv` y compararlos lado a lado (ver celda opcional al final de este paso).

In [ ]:
report_path = pathlib.Path(f"{quast_out}/report.tsv")

df = pd.read_csv(report_path, sep="\t", header=0, index_col=0)

# Tabla completa
print(f"=== Reporte QUAST completo (Caso {CASO}, {TECH}) ===")
print(df.to_string())

# Métricas clave para visualización
METRICAS_CLAVE = [
    "# contigs",
    "Largest contig",
    "Total length",
    "N50",
    "Genome fraction (%)",
    "Mismatches per 100 kbp",
]
print("\n=== Métricas clave ===")
print(df.loc[df.index.intersection(METRICAS_CLAVE)])

In [ ]:
# Gráfico de métricas clave de QUAST
# Seleccionamos métricas numéricas de interés para visualizar
METRICAS_PLOT = {
    "N50": "N50 (pb)",
    "Genome fraction (%)": "Fracción del genoma (%)",
    "# contigs": "Número de contigs",
    "Mismatches per 100 kbp": "Mismatches / 100 kbp",
}
col = df.columns[0]
valores = {label: float(df.loc[metric, col]) for metric, label in METRICAS_PLOT.items() if metric in df.index}

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(valores.keys(), valores.values(), color=["#4C72B0", "#55A868", "#C44E52", "#8172B2"])
ax.set_title(f"Métricas clave QUAST — Caso {CASO} ({TECH})")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

## 📦 Paso 7 — Comprimir todos los resultados

In [ ]:
all_results_archive = f"{workdir}/results/all_results.zip"

# Comprimir toda la carpeta de resultados
!zip -r "{all_results_archive}" "{workdir}/results"

print(f"\n✅ Todos los resultados comprimidos en: {all_results_archive}")
print("Puedes descargar este archivo desde el explorador de archivos de Colab.")

## ❓ Preguntas para reflexionar

1. ¿Cuántos contigs produjo su ensamblador (SPAdes o Flye)? ¿Cómo se compara con lo esperado para una bacteria de este tamaño?
2. ¿Cuál es el N50 y el L50 del ensamblaje? ¿Qué indican?
3. ¿Qué porcentaje del genoma de referencia fue cubierto (*Genome Fraction*)?
4. ¿Cuántos mismatches e indels/100 kbp reporta QUAST? ¿Cómo esperaría que cambien estos valores si comparara Illumina vs. Nanopore para el mismo organismo (dada la diferencia en tasa de error por base de cada tecnología)?
5. Si trabajó ambos bloques (`TECH`) para su caso: compare el número de contigs y el N50 entre el ensamblaje Illumina y el Nanopore. ¿Cuál quedó más contiguo (menos contigs, N50 más alto)? ¿Por qué las lecturas largas suelen ayudar a resolver regiones repetitivas que las lecturas cortas no pueden?
6. Su caso, ¿estaba marcado como "pesado" en la guía compartida? Si subsampleó con `seqtk`/`filtlong`, ¿cambió notablemente la calidad del ensamblaje frente a usar todos los datos? Justifique con las métricas de QUAST.
7. ¿Qué diferencia hay entre usar la referencia *durante* el ensamblaje y usarla *solo para evaluar* el resultado (como se hizo aquí con QUAST)?